## Rivers and Roads

In [ ]:
%load_ext line_profiler
%load_ext autoreload
%autoreload 2
%matplotlib qt

In [3]:
import numpy as np
from experimental import *
import matplotlib.pyplot as plt
import pandas as pd

import osmnx as ox
from shapely.ops import unary_union
from shapely.geometry import Polygon, MultiPolygon

In [4]:
from city2stl import roads

In [145]:
import napari

In [8]:
num_points = 100
x = np.linspace(-50, 50, num_points)
y = np.linspace(-50, 50, num_points)

# 2. Create the meshgrid
# xx and yy are 2D arrays containing all combinations of x and y coordinates
xx, yy = np.meshgrid(x, y)
im = xx**2 + yy**2
im = im - np.min(im)+1
im = im/1000000

In [11]:
# --- CONFIGURATION ---
CENTER = (39.9452, -75.1655)  # Philadelphia
DIST = 500                   # 1km radius



bbox = ox.utils_geo.bbox_from_point(CENTER, DIST )
bounds_NW = ((bbox[3],bbox[1]),(bbox[0],bbox[2]))


graph = ox.graph_from_point(CENTER, dist=DIST, network_type='drive')
gdf_roads = ox.graph_to_gdfs(graph, nodes=False)


models = {}
models["roads"] = roads.get_road_model(gdf_roads, im, bounds_NW)

In [ ]:

# --- 2. PROJECTION (To Meters for accurate math) ---
# We use EPSG:3857 (Web Mercator) so 1 unit = 1 meter
print("Processing roads...")
# Fetch Roads
graph = ox.graph_from_point(CENTER, dist=DIST, network_type='drive')


# --- 5. PROCESSING WATER ---

print("Processing water...")
gdf_water = ox.features_from_point(CENTER, {'natural': 'water', 'waterway': True}, dist=DIST)
gdf_water = gdf_water.to_crs(epsg=3857)
merged_water = unary_union(gdf_water.geometry.values).simplify(TOLERANCE_M)
water_polygons = geom_to_points(merged_water)
gdf_water = gdf_water.to_crs(epsg=4326)



In [13]:
import napari

In [14]:
print("opening - Napari")
v = napari.current_viewer()
if v is None: v = napari.Viewer()
v.layers.clear()

for key in models:
	print(key)

	vertices, faces = models[key]
	print(len(faces) )
	surface = (vertices,faces)
	s = v.add_surface(surface)
	s.wireframe.visible = True#len(faces) < 20000
	s.name = key
	s.opacity = 0.8
	s.blend_mode = "translucent"


opening - Napari


Invalid schema for package 'ome-types', please run 'npe2 validate ome-types' to check for manifest errors.


roads
10416


In [ ]:
tolerance_m = 0.5

# --- 4. VISUALIZATION ---
fig, ax = plt.subplots(figsize=(15, 15))
ax.set_facecolor('#111111') # Dark theme background

# Layer 1: Water (Bottom)
if not gdf_water.empty:
    gdf_water.plot(ax=ax, color='#1e3a5f', edgecolor='none', zorder=1)

# Layer 2: Roads (Middle)
# We simplify and merge roads for a cleaner look
merged_roads = unary_union(gdf_roads.geometry.values).simplify(tolerance_m)
pd.Series([merged_roads]).plot(ax=ax, color='#2a2a2a', edgecolor='none', alpha=0.9, zorder=2)

# Layer 3: Buildings (Top)
# Color buildings by their height attribute
gdf_buildings.plot(
    ax=ax,
    column='height_m',
    cmap='magma',
    edgecolor='#000000',
    linewidth=0.2,
    alpha=0.95,
    legend=True,
    legend_kwds={'label': "Building Height ($m$)", 'orientation': "horizontal", 'pad': 0.02, 'shrink': 0.6},
    zorder=3
)

# Styling
ax.set_axis_off()
plt.title(f"Urban Footprint: Philadelphia Core\n$1000m$ Radius at {center}", 
          color='white', fontsize=18, fontweight='bold', pad=-30)

plt.tight_layout()
plt.savefig('city_map_render.png', dpi=300, facecolor='#111111')
plt.show()

In [ ]:
fig, axs = plt.subplots(figsize=(30, 15), ncols=2, nrows=1, sharex=True, sharey=True, constrained_layout=True)


# Plot Buildings colored by height
gdf_buildings.plot(
    ax=axs[0], 
    column='height', 
    cmap='jet', 
    alpha=0.9, 
    edgecolor='#FFFFFF', 
    linewidth=2,
    zorder=2
)

# Formatting
axs[0].set_facecolor('#000000') # Dark background for a "pro" look



# Plot Buildings colored by height
gdf_buildings_optimized.plot(
    ax=axs[1], 
    column='height', 
    cmap='jet', 
    alpha=0.9, 
    edgecolor='#FFFFFF', 
    linewidth=2,
  
    zorder=2
)

# Formatting
axs[1].set_facecolor('#000000') # Dark background for a "pro" look


In [ ]:

if 0:

	# --- 7. FINAL DATA STRUCTURE ---
	# This is what you would pass to your renderer
	city_data = {
		"roads": road_polygons,      # List of [[x],[y]]
		"water": water_polygons,    # List of [[x],[y]]
		"buildings": building_layers # List of {"height": h, "points": [[x],[y]]}
	}

	print(f"Extraction Complete!")
	print(f"- Buildings layers: {len(building_layers)}")
	print(f"- Road polygons: {len(road_polygons)}")
	print(f"- Water polygons: {len(water_polygons)}")



In [ ]:
gdf = ox.features.features_from_point(center, tags, dist=dist)
bounds_NW = get_bbox(gdf)

## Merge Building polygons

In [ ]:
import numpy as np
from shapely.ops import triangulate


def extrude_gdf_to_mesh(gdf):
    all_vertices = []
    all_faces = []
    v_offset = 0

    for _, row in gdf.iterrows():
        geom = row.geometry
        height = row['height_m']
        
        # Get 2D coordinates of the exterior
        # We drop the last point because it is a duplicate of the first in OSM/Shapely
        coords_2d = np.array(geom.exterior.coords)[:-1]
        n = len(coords_2d)
        
        # 1. Create Vertices
        # Floor (z=0)
        floor_v = np.column_stack((coords_2d, np.zeros(n)))
        # Roof (z=height)
        roof_v = np.column_stack((coords_2d, np.full(n, height)))
        
        all_vertices.extend(floor_v)
        all_vertices.extend(roof_v)
        
        # 2. Create Wall Faces (Side surfaces)
        for i in range(n):
            next_i = (i + 1) % n
            # Two triangles per rectangular wall segment
            v1 = v_offset + i
            v2 = v_offset + next_i
            v3 = v_offset + n + next_i
            v4 = v_offset + n + i
            
            all_faces.append([v1, v2, v3])
            all_faces.append([v1, v3, v4])
            
        # 3. Create Roof Faces (Triangulation)
        # We triangulate the roof polygon to handle concave shapes
        # Note: simplify(0) helps ensure clean triangulation
        roof_poly = Polygon(coords_2d)
        # Using a simple ear-clipping or Delaunay triangulation for the roof cap
        # For complex shapes, you'd use a library like 'triangle' or 'mapbox_earcut'
        # Here's a basic manual triangulation approach for the roof:
        try:
            from shapely.ops import triangulate
            # This is a placeholder for a proper earcut; 
            # for 3D engines, 'triangles' are preferred over 'polygons'
            pass 
        except:
            pass
            
        v_offset += 2 * n

    return np.array(all_vertices), np.array(all_faces)

# Example Usage:
# vertices, faces = extrude_gdf_to_mesh(gdf_buildings_optimized)

## Get Rivers

In [ ]:
import osmnx as ox
import matplotlib.pyplot as plt
import pandas as pd

# 1. Configuration & Data Retrieval
center = (39.9452, -75.1655) 
dist = 1000

print("Fetching data from OSM...")
# Get Buildings
building_tags = {'building': True}
buildings_gdf = ox.features_from_point(center, building_tags, dist=dist)

# Get Roads (as a graph first, then to GDF for better topology)
graph = ox.graph_from_point(center, dist=dist, network_type='drive')
nodes_gdf, roads_gdf = ox.graph_to_gdfs(graph)

# 2. Projecting to Meters (EPSG:3857)
# This allows us to buffer roads using actual meter units
buildings_proj = buildings_gdf.to_crs(epsg=3857)
roads_proj = roads_gdf.to_crs(epsg=3857)

# 3. Processing Building Heights
# We check for building levels or height; default to 1 level (3m) if missing
if 'building:levels' in buildings_proj.columns:
    buildings_proj['render_height'] = pd.to_numeric(buildings_proj['building:levels'], errors='coerce').fillna(1) * 3
elif 'height' in buildings_proj.columns:
    buildings_proj['render_height'] = pd.to_numeric(buildings_proj['height'], errors='coerce').fillna(3)
else:
    buildings_proj['render_height'] = 3

# 4. Converting Road Lines to Polygons (Buffering)
# Assign widths based on highway type
width_map = {
    'motorway': 12,
    'trunk': 10,
    'primary': 8,
    'secondary': 7,
    'tertiary': 6,
    'residential': 4,
    'service': 2
}

def buffer_roads(row):
    # Default to 3 meters if road type is unknown
    width = width_map.get(row['highway'], 3) 
    return row.geometry.buffer(width)

print("Polygonizing roads...")
roads_proj['geometry'] = roads_proj.apply(buffer_roads, axis=1)

# 5. Visualization
print("Rendering map...")
fig, ax = plt.subplots(figsize=(15, 15))

# Plot Roads as Polygons
roads_proj.plot(ax=ax, color='#2c3e50', edgecolor='#34495e', linewidth=0.5, zorder=1)

# Plot Buildings colored by height
buildings_proj.plot(
    ax=ax, 
    column='render_height', 
    cmap='magma', 
    alpha=0.9, 
    edgecolor='#222222', 
    linewidth=0.3,
    legend=True, 
    legend_kwds={'label': "Estimated Height (m)", 'orientation': "horizontal", 'pad': 0.01},
    zorder=2
)

# Formatting
ax.set_facecolor('#1a1a1a') # Dark background for a "pro" look
ax.set_axis_off()
plt.title(f"City Footprint: {center}", color='white', fontsize=18, pad=-20)

plt.tight_layout()
plt.show()

In [ ]:
for key in models:
	vertices, faces = models[key]
	print(len(vertices) )
	
	print(len(faces) )

## SIMPILFY MODELS

## Visualize

In [ ]:

#s.normals.face.visible = True
#s.normals.vertex.visible = True

all_xy = []

for key in models:
    vertices, _ = models[key]
    vertices = np.asarray(vertices, float).reshape(-1,3)
    all_xy.append(vertices[:, :2])

all_xy = np.vstack(all_xy)
global_min_xy = all_xy.min(axis=0)
global_max_xy = all_xy.max(axis=0)
scale = GRID_SIZE / max(global_max_xy - global_min_xy)
depth_map_all = np.full((GRID_SIZE, GRID_SIZE), np.nan, dtype=np.float64)

for key in list(models.keys()):

	vertices, faces = models[key]
	vertices = np.asarray(vertices, float)
	faces = np.asarray(faces, int)

	xy = (vertices[:, :2] - global_min_xy) * scale
	z  = vertices[:, 2]
	xy = np.clip(xy, 0, GRID_SIZE-1)

	for f in faces:

		tri_xy = xy[f]
		tri_z  = z[f]
		
		if not check_triangle(tri_xy, tri_z):
			continue

		rasterize_triangle_max_edge(depth_map_all, tri_xy, tri_z)


plt.close('all')
plt.imshow(np.ma.masked_invalid(depth_map_all), cmap='viridis')
plt.colorbar(label='Max Z depth')
plt.title('Max-Z Depth Map (Edge-Function Rasterization)')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
for keys in models:

	vertices, faces = models[keys]
	surface = (vertices,faces)
	s = v.add_surface(surface)
	s.wireframe.visible = False

	mesh = trimesh.Trimesh(vertices,faces)
	mesh.export("Granada_" +keys+".stl")


In [ ]:
vertices2 = []
faces2 = []
maxfaces = 0

for keys in models:

	vertices, faces = models[keys]

	faces = faces + maxfaces
	print(faces.max())
	print(len(vertices))
	vertices2.append(vertices)
	faces2.append(faces)

	maxfaces = maxfaces + (faces.max()+1)

vertices2 = np.concatenate(vertices2,axis=0)
faces2    = np.concatenate(faces2, axis=0)

In [ ]:
mesh = trimesh.Trimesh(vertices2,faces2)
mesh.export("Granada_together.stl")

In [ ]:
from skimage import filters,transform
 
outshape = np.array(data.shape)*2
filt = transform.resize(data, outshape)
filt = filters.gaussian( filt , sigma=3, truncate=3)
filt = filters.median(filt, np.ones((3,3)))

plt.figure()
plt.imshow(filt)

#data = dem.data
facet = np2stl.numpy2stl(filt)
solid = np2stl.Solid(facet)
#solid.simplify()
fn = "granada_topo.stl" 
solid.save_stl(fn)

In [ ]:
x = 660
y = 1470
plt.imshow(im[-780:-570,1370:1570])

In [ ]:
r0 = 2820
r1 = 3030
c0 = 1370
c1 = 1570

mx = im[r0:r1, c0:c1]
plt.imshow(mx)

In [ ]:
plt.imshow(im)
plt.plot(Wcoor,Ncoor, "ro")

In [ ]:
ax = plt.figure().add_subplot(projection='3d')
ax.plot(Ncoor,Wcoor,H, "o")

## Barranquilla

In [ ]:
#gdf["height"] = 30

gdf["heights"] = 30#building_heights(gdf)*.05
gdf["topo_base"] = 0
gdf["z0"] = gdf["topo_base"]
gdf["z1"] = gdf["z0"]+ gdf["heights"]

building_poly = get_polygons(gdf)
tris = triangulate_buildings(building_poly)

In [ ]:
vertices, faces = np2stl.vertices_to_index(tris)
vertices[:,[1,0]] = vertices[:,[0,1]]
vertices[:,2] = vertices[:,2] * scale
vertices = vertices*1000

In [ ]:
import napari
print("opening - Napari")
v = napari.current_viewer()
if v is None: v = napari.Viewer()
v.layers.clear()

surface = (vertices,faces)
s = v.add_surface(surface)
s.wireframe.visible = False

In [ ]:
import matplotlib.pyplot as plt

import osmnx as ox


In [ ]:

# Define the place or bounding box
n,s,e,w = (11.058,10.92,-74.75,-74.85)

G = ox.graph_from_bbox(north=n,south=s,east=e,west=w, network_type='all')
# Download the data for all roads
#G = ox.graph_from_place(place, network_type='all')


In [ ]:
dem = DEM(Root, (n,s,e,w))
demdata = dem.data

In [ ]:
x = dem.data.copy()
x = ndi.morphological_gradient(dem.data, (3,3))<0.001
x = 1.*x + 1.*(dem.data.clip(0,3)<0.15)

In [ ]:
x = rescale(x,sz_out=2000) 
x = x/x.max()

demdata = rescale(demdata,sz_out=2000)

demdata = demdata / demdata.max() * 10 
demdata = demdata-(x*2.)

In [ ]:
# Plot the graph at different zoom levels
fig, ax = plt.subplots(1, 2, figsize=(20, 10))

# Zoomed-out view (low zoom level)
ox.plot_graph(ox.project_graph(G), ax=ax[0], node_size=0, edge_color='grey', edge_linewidth=0.2)
ax[0].set_title("Zoomed-Out View")

plt.show()

In [ ]:
def graph2lines(G):

	rivers = []
	for u, v, data in G.edges(keys=False, data=True):
		
		name = ""
		if "name" in data:  
			name = data["name"]         

		if "geometry" in data: 
			pts = np.array(data["geometry"].xy)
		else:
			node = G.nodes
			pts = [[ node[u]['x'], node[v]['x']],
				[ node[u]['y'], node[v]['y']]]
			pts = np.array(pts)

		line = Line(name, pts, color="g", type="river")
		rivers.append(line)
	return rivers

In [ ]:
rivers2 = graph2lines(G)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(20, 10))

for line in rivers2:
    x,y = line.pts
    ax[0].plot(x,y,"g")
    
ax[0].grid(True)

In [ ]:
linepts = [line.pts for line in rivers2]

lineptsim = [coor2im(((n,s),(w,e)), ((0,demdata.shape[0]),(0,demdata.shape[1])), pts.T) 
             for pts in linepts]

lines_im = embed_lines(np.zeros_like(demdata).T, lineptsim , res=1).T

In [ ]:
building_im = morphology.area_opening(lines_im==0, 500) == 0
building_im = building_im & ~(lines_im)

In [ ]:
plt.imshow(building_im)

In [ ]:
landscape = demdata + building_im *5 + lines_im*-1 

In [ ]:
landscape = landscape - landscape.min() + 1

In [ ]:
savefile(out_dir, name, im2 )

In [ ]:
plt.imshow(landscape, cmap="jet")

In [ ]:
facet = np2stl.numpy2stl(landscape)
solid = np2stl.Solid(facet)
vx = solid.vertices.copy().astype(float)
########################


In [ ]:
np2stl.numpy2stl

In [ ]:
import napari
print("opening - Napari")
v = napari.current_viewer()
if v is None: v = napari.Viewer()
v.layers.clear()

surface = (vx,solid.faces)
s = v.add_surface(surface)
s.wireframe.visible = False

In [ ]:
import strm2stl.numpy2stl.numpy2stl as np3D

def savefile(out_dir, name, im2 ):
	if not os.path.isdir(out_dir): return

	out_dir = out_dir + name 
	os.makedirs(out_dir,exist_ok=True)
	
	tri = np3D.numpy2stl(im2)
	facets = np3D.triangles_to_facets(tri)
	print("Saving")
	np3D.writeSTL(facets, out_dir + "/" + name + ".stl")

In [ ]:
out_dir = "C:/Users/eac84/Desktop/Desktop/Tasks/STL/Maps/"

savefile(out_dir, "Barranquilla", landscape )

## SPA Race Track

In [ ]:
from osmnx import utils_geo

In [ ]:
#gdf = ox.features_from_place("Spa, Belgium", tags)

N,W = 50.438343, 5.970514
bbox = utils_geo.bbox_from_point((N,W), dist=2000)
n,s,e,w = bbox
dem = DEM(Root, (n,s,e,w))
G =  ox.graph_from_bbox( *bbox, network_type='all')
rivers2 = graph2lines(G)




In [ ]:
tags = {'building': True}
gdf = ox.features.features_from_bbox(*bbox, tags = tags)

In [ ]:
gdf

In [ ]:

fig, ax = plt.subplots(1, 1, figsize=(20, 10))

for line in rivers2:
    x,y = line.pts
    ax[0].plot(x,y,"g")
    
ax[0].grid(True)

In [ ]:
dem.draw()

In [ ]:

plt.figure()
for u, v, data in G.edges(keys=False, data=True):

	color = "g"     
	if "name" in data:  
		
		name = data["name"]
		color = "b"     
		print(name)
		if "Circuit" in name:
			color = "r"


	if 'highway' in data:
		if data["highway"] == "track":
			color = "m"
			continue

	if 'ref' in data: 
		color = "y"
		#'N62c'

	if "geometry" in data: 
		pts = np.array(data["geometry"].xy)
	else:
		node = G.nodes
		pts = [[ node[u]['x'], node[v]['x']],
			[ node[u]['y'], node[v]['y']]]
		pts = np.array(pts)

	plt.plot(pts[0],pts[1], color = color)			


In [ ]:
polygons = []
for n,row in enumerate(gdf.itertuples(index=False)):
	geometry = row.geometry

	if isinstance(geometry, Polygon):
		pts = np.array(geometry.exterior.xy)
		poly = {"points":pts}
		polygons.append(poly)
	elif isinstance(geometry, MultiPolygon):
		#if geometry is multipolygon, go through each subpolygon 
		for subpolygon in geometry.geoms: 
			pts = np.array(subpolygon.exterior.xy)
			poly = {"points":pts}
			polygons.append(poly)


In [ ]:
for p in polygons:
  x,y = p["points"]
  plt.fill(x,y)

In [ ]:
plt.grid(True)

In [ ]:
from strm2stl.create import coor2im

In [ ]:
demdata = rescale(demdata,sz_out=1000)

In [ ]:
linepts = [line.pts for line in rivers2]

lineptsim = [coor2im(((n,s),(w,e)), ((0,demdata.shape[0]),(0,demdata.shape[1])), pts.T) 
             for pts in linepts]

lines_im = embed_lines(demdata.T, lineptsim , res=1).T

In [ ]:
lines_im = 50*embed_lines(demdata.T, lineptsim , res=1).T + demdata

In [ ]:
out_dir = "C:/Users/eac84/Desktop/Desktop/Tasks/STL/Maps/"

lines_im = lines_im - lines_im.min() + 1
savefile(out_dir, "Spa", lines_im )

In [ ]:
plt.figure()
plt.imshow(lines_im )

In [ ]:
demdata.min(), demdata.max()